## 2️⃣ Création de la Base PostgreSQL & 3️⃣ Chargement des Données via SQLAlchemy



# connexion database

In [ ]:
import pandas as pd 
import os 
from dotenv import load_dotenv
from sqlalchemy import engine,create_engine,text
DB_USER=os.getenv("DB_USER")
DB_PASS=os.getenv("DB_PASS")
DB_HOST=os.getenv("DB_HOST")
DB_PORT=os.getenv("DB_PORT")
DB_NAME=os.getenv("DB_NAME")

load_dotenv()
try:
    db_url=f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine=create_engine(db_url)
    with engine.connect()as con:
        rs=con.execute(text("select version ();"))
        db_ver=rs.fetchone()
    print(f"conexion a data base creer ! {db_ver}")
except  Exception as e:
    print(f"error connexion a data base ! {e}")

# creation table 

In [ ]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS segment(
        segment_id SERIAL PRIMARY KEY,
        segment_client VARCHAR(60) UNIQUE NOT NULL
    );
         CREATE TABLE IF NOT EXISTS client(
        client_id VARCHAR(60) PRIMARY KEY,
        score_credit_client DECIMAL(10,3),
        segment_id INT REFERENCES segment(segment_id)
    );
        CREATE TABLE IF NOT EXISTS agence(
        agence_id SERIAL PRIMARY KEY,
        agence VARCHAR(60) UNIQUE NOT NULL
    );
        CREATE TABLE IF NOT EXISTS produit(
        produit_id SERIAL PRIMARY KEY,
        produit VARCHAR(60),
        categorie VARCHAR(60)
    );
        CREATE TABLE IF NOT EXISTS transactions(
        transaction_id VARCHAR(60) PRIMARY KEY,
        client_id VARCHAR(60) REFERENCES client(client_id),
        produit_id INT REFERENCES produit(produit_id),
        agence_id INT REFERENCES agence(agence_id),
        date_transaction VARCHAR(60),
        montant DECIMAL(12,2),
        devise VARCHAR(10),
        taux_change_eur DECIMAL(10,4),
        type_operation VARCHAR(60),
        statut VARCHAR(30),
        solde_avant DECIMAL(12,2),
        is_anomalie VARCHAR(60)
    );   
    """))

# chargement_donner

In [ ]:
df=pd.read_csv(r"C:\Users\AMINE JBR\Documents\d\data\financecore_clean.csv")
df.head()

# inserer les donner 

In [ ]:
with engine.begin() as con :
    segment=df["segment_client"].drop_duplicates()
    segment.to_sql(name="segment",index=False,if_exists="append",con=con)
    segment_db=pd.read_sql(text("select*from segment"),con=con)

In [ ]:
df=pd.merge(df,segment_db,on="segment_client",how="left")

In [ ]:
client=df[["client_id","score_credit_client","segment_id"]].drop_duplicates(subset=["client_id"])

In [ ]:
with engine.begin() as con:
    client.to_sql(name="client",index=False,if_exists="append",con=con) 

In [ ]:
with engine.begin() as con :
    agence=df["agence"].drop_duplicates()
    produit=df[["produit","categorie"]].drop_duplicates()
    agence.to_sql(name="agence",if_exists="append",index=False,con=con)
    produit.to_sql(name="produit",if_exists="append",index=False,con=con)


In [ ]:
with engine.begin() as con :
    agence_db=pd.read_sql(text("select*from agence"),con)
    produit_db=pd.read_sql(text("select*from produit"),con)
    df=pd.merge(df,agence_db,on="agence",how="left")
    df=pd.merge(df,produit_db,on=["produit","categorie"],how="left")


In [ ]:
with engine.begin() as con :
    transactions=df[['transaction_id', 'client_id', 'produit_id', 'agence_id','date_transaction', 'montant', 'devise', 'taux_change_eur','type_operation', 'statut', 'solde_avant', 'is_anomalie']].drop_duplicates(subset=["transaction_id"])
    transactions.to_sql(name="transactions",if_exists='append',index=False,con=con)

In [ ]:
with engine.begin() as con:
    transaction=pd.read_sql(text("select*from transactions"),con)
transaction.columns

In [ ]:
df.columns

# creation index 

In [ ]:
with engine.begin() as con:
    con.execute(text("CREATE INDEX idx_client_id ON transactions(client_id)"))
    con.execute(text("CREATE INDEX idx_date_transactions ON transactions(date_transaction)"))
    con.execute(text("CREATE INDEX idx_agence  ON transactions(agence_id)"))



# creation vues jointure

In [ ]:
with engine.begin() as con :
    con.execute(text("""
    CREATE  OR REPLACE VIEW jointure_table as 
    SELECT 
    t.transaction_id,
    t.montant ,
    t.devise,
    t.taux_change_eur,
    t.type_operation,
    t.statut,
    t.solde_avant,
    t.is_anomalie,
    cl.client_id,
    cl.score_credit_client,
    p.produit,
    p.categorie,
    a.agence,
    s.segment_client
    from transactions t 
    join client cl ON t.client_id=cl.client_id
    join segment s ON s.segment_id=cl.segment_id
    join produit p ON t.produit_id=p.produit_id
    join agence a ON t.agence_id = a.agence_id 
    """))

# Vérifier l'intégrité référentielle

In [ ]:
with engine.begin() as con:
    result_client=con.execute(text(""" select count(*) as orphan_client 
        from client cl
        LEFT JOIN transactions t 
        ON t.client_id = cl.client_id
        WHERE t.client_id IS NULL
    """))
    result_segment=con.execute(text(""" SELECT count(*) as orphan_segment
        FROM segment s
        LEFT JOIN client cl 
        ON s.segment_id=cl.segment_id
        WHERE cl.segment_id IS NULL
    """))
    result_produit=con.execute(text("""SELECT count(*) AS orphan_produit
        FROM produit p 
        LEFT JOIN transactions t
        ON p.produit_id=t.produit_id
        WHERE t.produit_id IS NULL
    """))
    result_agence=con.execute(text("""SELECT count(*) AS orphan_agence
        FROM agence a 
        LEFT JOIN transactions t
        ON a.agence_id=t.agence_id
        WHERE t.agence_id IS NULL
    """))
print(result_client.fetchone())
print(result_segment.fetchone())
print(result_produit.fetchone())
print(result_agence.fetchone())


In [ ]:
with engine.begin() as con:
    result = pd.read_sql(text("""
        SELECT
            t.agence_id,
            t.produit_id,
            DATE_TRUNC('month', t.date_transaction::DATE) AS mois,
            SUM(t.montant) AS total_transactions,
            AVG(t.montant) AS moyenne_transactions
        FROM transactions t
        GROUP BY
            t.agence_id,
            t.produit_id,
            DATE_TRUNC('month', t.date_transaction::DATE)
        HAVING SUM(t.montant) > 1000
    """), con)

print(result)

In [ ]:
with engine.begin() as con:
    result = pd.read_sql(text("""
        SELECT
            client_id,
            score_credit_client
        FROM client
        WHERE score_credit_client < (
            SELECT AVG(score_credit_client)
            FROM client
        )
    """), con)

result

In [ ]:
with engine.begin() as con:
    result = pd.read_sql(text("""
        SELECT
            segment_id,
            COUNT(*) AS total_clients,

            SUM(
                CASE 
                    WHEN score_credit_client < 500 THEN 1
                    ELSE 0
                END
            ) AS nb_defaut,

            AVG(
                CASE 
                    WHEN score_credit_client < 500 THEN 1.0
                    ELSE 0.0
                END
            ) AS taux_defaut

        FROM client
        GROUP BY segment_id
    """), con)

result


In [ ]:
with engine.begin() as con:
    result = pd.read_sql(text("""
        SELECT
            t.transaction_id,
            t.date_transaction,
            t.montant,

            c.client_id,
            c.score_credit_client,

            s.segment_id,

            p.produit,

            a.agence_id

        FROM transactions t

        LEFT JOIN client c
            ON t.client_id = c.client_id

        LEFT JOIN segment s
            ON c.segment_id = s.segment_id

        LEFT JOIN produit p
            ON t.produit_id = p.produit_id

        LEFT JOIN agence a
            ON t.agence_id = a.agence_id
    """), con)

result

In [ ]:
with engine.begin() as con:
    con.execute(text("""
        CREATE OR REPLACE VIEW kpi_total_transactions AS
        SELECT
            COUNT(*) AS nb_transactions,
            SUM(montant) AS total_montant,
            AVG(montant) AS moyenne_montant
        FROM transactions;
    """))

In [ ]:
with engine.begin() as con:
    con.execute(text("""
        CREATE OR REPLACE VIEW kpi_total_transactions AS
        SELECT
            COUNT(*) AS nb_transactions,
            SUM(montant) AS total_montant,
            AVG(montant) AS moyenne_montant
        FROM transactions;
    """))

In [ ]:
with engine.begin() as con:
    result=con.execute(text("""
        CREATE OR REPLACE VIEW kpi_risk_segment AS
        SELECT
            segment_id,
            COUNT(*) AS nb_clients,
            AVG(score_credit_client) AS score_moyen,
            AVG(
                CASE 
                    WHEN score_credit_client < 500 THEN 1.0
                    ELSE 0.0
                END
            ) AS taux_defaut
        FROM client
        GROUP BY segment_id;
    """))

In [ ]:
with engine.begin() as con:
    con.execute(text("""
        CREATE OR REPLACE VIEW kpi_transactions_mensuelles AS
        SELECT
            DATE_TRUNC('month', date_transaction::DATE) AS mois,
            COUNT(*) AS nb_transactions,
            SUM(montant) AS total_montant
        FROM transactions
        GROUP BY DATE_TRUNC('month', date_transaction::DATE);
    """))